# Collector de Futuros de Café (KC) — ICE/NYBOT

Descarga toda la curva de futuros de Coffee C desde Interactive Brokers (TWS) y la guarda en `data_kc.json`.

**Requisitos:**
- TWS corriendo en `127.0.0.1:7496`
- `pip install ib_async`

**Contratos:** KC + H (Mar), K (May), N (Jul), U (Sep), Z (Dec) + año 2 dígitos (e.g. KCN26, KCZ27)

In [ ]:
# CAFE (KC) - ACTUALIZACIÓN DE PRECIOS DESDE INTERACTIVE BROKERS
from ib_async import IB, Future
from pathlib import Path
from datetime import datetime, timedelta, date
import pandas as pd
import json, re, os, tempfile
import asyncio

# ========= CONFIG =========
HOST = "127.0.0.1"
PORT = 7496        # 7497 (paper) / 7496 (live)
CLIENT_ID = 20     # Usar un ID distinto a los demas collectors

JSON_PATH = Path(
    r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents"
    r"\Super de Alimentos\Gestión de riesgo\Data\data_kc.json"
)

# FECHA MÍNIMA - Solo traer datos desde esta fecha
START_DATE = "2025-01-01"
# ==========================

# ICE Coffee C: meses H(Mar), K(May), N(Jul), U(Sep), Z(Dec)
MONTH_LETTERS = {'H': '03', 'K': '05', 'N': '07', 'U': '09', 'Z': '12'}
MONTH_NUM_TO_LETTER = {v: k for k, v in MONTH_LETTERS.items()}
ORDERED_MONTHS = ['H', 'K', 'N', 'U', 'Z']

# ---------- util contrato ----------
def code_to_yyyymm(code: str) -> str:
    m = re.fullmatch(r"KC([HKNUZ])(\d{2})", code.strip().upper())
    if not m:
        raise ValueError(f"Código inválido: {code}")
    letter, yy = m.groups()
    yy_i = int(yy)
    year = 2000 + yy_i if yy_i <= 79 else 1900 + yy_i
    return f"{year}{MONTH_LETTERS[letter]}"

def yyyymm_to_code(yyyymm: str) -> str:
    y = int(yyyymm[:4])
    mm = yyyymm[4:6]
    return f"KC{MONTH_NUM_TO_LETTER[mm]}{str(y)[-2:]}"

async def qualify_kc_async(ib: IB, yyyymm: str):
    for ex in ("NYBOT", "ICEUS"):
        fut = Future(
            symbol="KC",
            lastTradeDateOrContractMonth=yyyymm,
            exchange=ex,
            currency="USD",
            includeExpired=True
        )
        try:
            qs = await ib.qualifyContractsAsync(fut)
            if qs:
                return qs[0]
        except Exception:
            pass
        await asyncio.sleep(0.05)
    return None

# ---------- históricos ----------
async def fetch_history_from_date(ib: IB, q, start_date: str):
    """Descarga histórico desde start_date hasta hoy"""
    start_dt = pd.to_datetime(start_date)
    end_dt = datetime.now()
    days = max(1, (end_dt - start_dt).days + 1)
    
    for what in ("TRADES", "BID_ASK", "MIDPOINT"):
        try:
            bars = await ib.reqHistoricalDataAsync(
                q, endDateTime="", durationStr=f"{days} D",
                barSizeSetting="1 day", whatToShow=what,
                useRTH=False, formatDate=2
            )
            if bars:
                rows = []
                for b in bars:
                    d = str(b.date)
                    if d >= start_date:
                        rows.append({
                            "date": d, "open": b.open, "high": b.high, "low": b.low,
                            "close": b.close, "volume": b.volume, "openinterest": None
                        })
                if rows:
                    df = pd.DataFrame(rows).drop_duplicates("date").sort_values("date")
                    return df.to_dict(orient="records")
        except Exception:
            pass
        await asyncio.sleep(0.25)
    return []

# ---------- JSON I/O ----------
def load_db(path: Path) -> dict:
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def atomic_save(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name, dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)
    except Exception:
        try:
            os.remove(tmp)
        finally:
            raise

# ---------- descubrir contratos ----------
async def discover_contracts(ib: IB, max_years_ahead: int = 3) -> list:
    """Descubre contratos activos de KC buscando desde hoy hasta max_years_ahead años."""
    today = date.today()
    found = []
    for year_offset in range(max_years_ahead + 1):
        y = today.year + year_offset
        for letter in ORDERED_MONTHS:
            yyyymm = f"{y}{MONTH_LETTERS[letter]}"
            code = f"KC{letter}{str(y)[-2:]}"
            q = await qualify_kc_async(ib, yyyymm)
            if q:
                found.append(code)
                print(f"  ✅ {code} ({yyyymm})")
            await asyncio.sleep(0.1)
    return found

# ---------- main async ----------
async def main_async():
    db = load_db(JSON_PATH)
    
    ib = IB()
    try:
        await ib.connectAsync(HOST, PORT, clientId=CLIENT_ID)
    except Exception as e:
        print("No se pudo conectar a IBKR:", e)
        return

    try:
        today = datetime.now()
        
        # Descubrir contratos si el JSON esta vacio
        kc_contracts = [k for k in db.keys() if re.fullmatch(r"KC[HKNUZ]\d{2}", k)]
        if not kc_contracts:
            print("JSON vacío — descubriendo contratos activos en IBKR...")
            kc_contracts = await discover_contracts(ib)
            for code in kc_contracts:
                if code not in db:
                    db[code] = []
            print(f"\nDescubiertos: {len(kc_contracts)} contratos\n")
        else:
            print(f"Contratos en JSON: {len(kc_contracts)}")
        
        print(f"Fecha mínima: {START_DATE}\n")
        
        updated = 0
        
        for code in sorted(kc_contracts):
            try:
                yyyymm = code_to_yyyymm(code)
            except ValueError:
                continue
            
            q = await qualify_kc_async(ib, yyyymm)
            if not q:
                print(f"⚠️ {code}: no calificado en IBKR")
                continue
            
            bars = db.get(code, [])
            bars = [r for r in bars if r["date"] >= START_DATE]
            last_date = max((r["date"] for r in bars), default=None)
            
            if last_date:
                start = (pd.to_datetime(last_date) + timedelta(days=1)).strftime("%Y-%m-%d")
                if start <= today.strftime("%Y-%m-%d"):
                    new_rows = await fetch_history_from_date(ib, q, start)
                    if new_rows:
                        merged = (pd.concat([pd.DataFrame(bars), pd.DataFrame(new_rows)], ignore_index=True)
                                  .drop_duplicates("date").sort_values("date"))
                        db[code] = merged.to_dict(orient="records")
                        print(f"✅ {code}: +{len(new_rows)} filas (hasta {merged.iloc[-1]['date']})")
                        updated += 1
                    else:
                        print(f"   {code}: al día ({last_date})")
                else:
                    print(f"   {code}: al día ({last_date})")
            else:
                rows = await fetch_history_from_date(ib, q, START_DATE)
                if rows:
                    db[code] = rows
                    print(f"✅ {code}: {len(rows)} filas (desde {START_DATE})")
                    updated += 1
                else:
                    print(f"⚠️ {code}: sin datos")
            
            await asyncio.sleep(0.5)
        
        atomic_save(JSON_PATH, db)
        
        print(f"\n{'='*50}")
        print(f"Actualizados: {updated} | Archivo: {JSON_PATH.name}")
        print(f"{'='*50}")
        
        print("\nÚltima fecha por contrato:")
        for code in sorted([k for k in db.keys() if re.fullmatch(r'KC[HKNUZ]\d{2}', k)],
                           key=lambda x: code_to_yyyymm(x)):
            bars = db.get(code, [])
            last = max((r["date"] for r in bars), default="N/A")
            print(f"  {code}: {last}")

    finally:
        if ib.isConnected():
            ib.disconnect()

await main_async()

In [ ]:
# SUBIR A SUPABASE - Ejecutar después de actualizar el JSON
import sys
sys.path.insert(0, r'C:\Users\DanielAristizabal\Documents\GitHub\pysdk')

from dotenv import load_dotenv
load_dotenv(r'C:\Users\DanielAristizabal\Documents\GitHub\pysdk\.env')

from gestion_de_riesgos.collectors.base_collector import collect_all, collect_all_contracts

# 1) Front contract → risk_prices (para VaR y Benchmark)
result1 = collect_all('2025-01-01', '2026-12-31')
print('collect_all:', result1)

# 2) Todos los contratos → risk_prices_all_contracts (para Portafolio GR mark-to-market)
result2 = collect_all_contracts('2025-01-01', '2026-12-31')
print('collect_all_contracts:', result2)